In [1]:
# ── Core libraries used by every tool section ─────────────────
%pip install pymupdf jiwer pandas numpy Levenshtein pillow --quiet
print("✅ Core libraries ready")

Note: you may need to restart the kernel to use updated packages.
✅ Core libraries ready


In [5]:
import os, sys, json, csv, warnings, textwrap
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage

warnings.filterwarnings("ignore")

# ── Load API keys from .env ────────────────────────────────────
load_dotenv()   # reads .env in the project root

# ── Project root ───────────────────────────────────────────────
ROOT = Path().resolve()
if not (ROOT / "adapters").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CORPUS_DIR       = ROOT / "corpus"
GROUND_TRUTH_DIR = ROOT / "ground_truth"
OUTPUT_DIR       = ROOT / "outputs"
IMG_DIR          = OUTPUT_DIR / "images"

OUTPUT_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

# ── Discover PDFs dynamically ─────────────────────────────────
PDF_FILES = sorted(CORPUS_DIR.glob("*.pdf"))

print(f"Root : {ROOT}")
print(f"PDFs found: {[p.name for p in PDF_FILES]}")
if not PDF_FILES:
    print("⚠️  No PDFs in corpus/ — drop your files there and re-run this cell")


Root : /workspaces/PDF_Data_Extraction
PDFs found: ['1_Rapport-financier-semestriel-AFD-2025.pdf', '2_perfect-pitch-deck.pdf', '3_NVCA-Model-Voting-Agreement.pdf', '4_SA-war-scanned.pdf', '5_2025-integrated-annual-report.pdf', '6_ibm-annual-report-2025.pdf']


In [6]:
# ════════════════════════════════════════════════════════════════
# SHARED EVALUATION FUNCTIONS
# Used immediately after every tool — do not skip this cell.
# ════════════════════════════════════════════════════════════════
import Levenshtein
from jiwer import cer as _jiwer_cer

# ── CER — Character Error Rate ────────────────────────────────
def compute_cer(hypothesis: str, reference: str) -> float:
    """
    CER = edit_distance(hyp, ref) / len(ref)   at character level.
    Returns 0.0 (perfect) → 1.0+ (very poor).
    """
    if not reference.strip():
        return 0.0
    return round(_jiwer_cer(reference.strip(), hypothesis.strip()), 4)

def cer_to_score5(cer: float) -> float:
    """Map CER → 1-5 benchmark scale (5 = best)."""
    return round(max(1.0, (1 - cer) * 5), 2)

# ── TEDS — Tree Edit Distance Similarity ─────────────────────
def _table_to_tree(table: list) -> str:
    """Linearise a table as a bracketed tree string."""
    rows = "".join(
        "(row " + "".join(f"(cell {str(c).strip()})" for c in row) + ")"
        for row in table
    )
    return f"(table {rows})"

def compute_teds(pred_table: list, gt_table: list) -> float:
    """
    Simplified TEDS via normalised Levenshtein on linearised table trees.
    Returns 0.0 (no match) → 1.0 (perfect).
    """
    if not gt_table:   return 1.0
    if not pred_table: return 0.0
    p = _table_to_tree(pred_table)
    g = _table_to_tree(gt_table)
    dist  = Levenshtein.distance(p, g)
    max_l = max(len(p), len(g))
    return round(1.0 - dist / max_l, 4) if max_l else 1.0

def teds_to_score5(teds: float) -> float:
    return round(teds * 5, 2)
"""
# ── Ground-truth loader ───────────────────────────────────────
def load_gt(pdf_stem: str) -> dict:
    
    gt_path = GROUND_TRUTH_DIR / pdf_stem / "metadata.json"
    if not gt_path.exists():
        return {}
    meta  = json.loads(gt_path.read_text("utf-8"))
    pages = {}
    for p_str, ann in meta.get("annotations", {}).items():
        p        = int(p_str)
        txt_file = GROUND_TRUTH_DIR / pdf_stem / ann["text_file"]
        csv_file = GROUND_TRUTH_DIR / pdf_stem / ann["table_file"]
        pages[p] = {
            "text"  : txt_file.read_text("utf-8") if txt_file.exists() else "",
            "tables": [],
        }
        if csv_file.exists():
            with open(csv_file, newline="", encoding="utf-8") as f:
                pages[p]["tables"] = [list(csv.reader(f))]
    return pages """

# ── Evaluation printer ────────────────────────────────────────
def print_eval(tool: str, pdf_stem: str, hyp_text: str, pred_tables: list):
    """Run CER + TEDS against all annotated pages and print a result table."""
    gt = load_gt(pdf_stem)
    if not gt:
        print(f"  ⚠  No ground truth for '{pdf_stem}'")
        print(f"     Run: python ground_truth/prepare_ground_truth.py "
              f"--pdf corpus/{pdf_stem}.pdf --id {pdf_stem} --pages 1 2 3")
        return
    print(f"\\n  📊  {tool.upper()} — {pdf_stem}")
    print(f"  {'Page':>4}  {'CER':>6}  {'CER/5':>6}  {'TEDS':>6}  {'TEDS/5':>7}")
    print(f"  {'─'*4}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*7}")
    for page_num, gt_page in sorted(gt.items()):
        cer  = compute_cer(hyp_text, gt_page["text"])
        teds_scores = [
            compute_teds(pt, gt_page["tables"][0])
            for pt in pred_tables
        ] if gt_page["tables"] and pred_tables else []
        teds = sum(teds_scores) / len(teds_scores) if teds_scores else None
        t_str  = f"{teds:.4f}" if teds is not None else "  N/A "
        t5_str = f"{teds_to_score5(teds):.2f}" if teds is not None else "  N/A"
        print(f"  {page_num:>4}  {cer:>6.4f}  {cer_to_score5(cer):>6.2f}  {t_str:>6}  {t5_str:>7}")
    print()
    print("  CER  : 0.00=perfect  <0.05=excellent  <0.15=good  >0.30=poor")
    print("  TEDS : 1.00=perfect  >0.85=excellent  >0.70=good  <0.50=poor")

print("✅ Shared evaluation functions loaded")


✅ Shared evaluation functions loaded


In [8]:
import pandas as pd

pdf_path = PDF_FILES[0]   # ← change index or set to a specific Path
pdf_stem = pdf_path.stem
result   = {VAR}[pdf_stem]

print("=" * 62)
print(f"Tool  : {TOOL_NAME}")
print(f"PDF   : {pdf_stem}  ({result['pages']} pages)")
print(f"Time  : {result['elapsed']:.1f}s   "
      f"RAM Δ: {result['memory_mb']:.0f} MB   "
      f"Cost : ${result['cost']:.4f}")

# ── Text ──────────────────────────────────────────────────────
print("\\n── TEXT (first 800 chars) " + "─"*35)
print(textwrap.fill(result["text"][:800], width=80))

# ── Tables ────────────────────────────────────────────────────
print(f"\\n── TABLES  ({len(result['tables'])} found) " + "─"*38)
for i, tbl in enumerate(result["tables"][:3]):
    print(f"\\n  Table {i+1}  ({len(tbl)} rows × {len(tbl[0]) if tbl else 0} cols)")
    print(pd.DataFrame(tbl).to_string(index=False, header=False, max_rows=6))

# ── Images ────────────────────────────────────────────────────
print(f"\\n── IMAGES  ({len(result.get('image_paths',[]))} saved) " + "─"*36)
for p in result.get("image_paths", [])[:3]:
    print(f"  {p}")
    try:
        display(IPImage(filename=p, width=280))
    except Exception:
        pass

NameError: name 'VAR' is not defined

In [9]:
for pdf_path in PDF_FILES:
    pdf_stem = pdf_path.stem
    if pdf_stem not in {VAR}:
        continue
    result = {VAR}[pdf_stem]
    print_eval("{TOOL_NAME}", pdf_stem,
               hyp_text    = result["text"],
               pred_tables = result["tables"])


NameError: name 'VAR' is not defined

In [10]:
def preview_cell(var, tool):
    return code(PREVIEW_CELL.replace("{VAR}", var).replace("{TOOL_NAME}", tool))

def eval_cell(var, tool):
    return code(EVAL_CELL.replace("{VAR}", var).replace("{TOOL_NAME}", tool))

In [11]:
%pip install pymupdf4llm pymupdf --quiet
print("✅ PyMuPDF4LLM ready")


Note: you may need to restart the kernel to use updated packages.
✅ PyMuPDF4LLM ready


In [12]:
import time, fitz, pymupdf4llm, psutil

def run_pymupdf4llm(pdf_path: Path) -> dict:
    proc = psutil.Process()
    mem0 = proc.memory_info().rss / (1024**2)
    t0   = time.perf_counter()

    # ── Markdown (primary output) ─────────────────────────────
    markdown = pymupdf4llm.to_markdown(str(pdf_path))

    # ── Page-by-page: plain text + tables + images ────────────
    doc         = fitz.open(str(pdf_path))
    plain_parts = []
    tables      = []
    image_paths = []

    for page_num, page in enumerate(doc, start=1):
        plain_parts.append(page.get_text("text"))

        # Tables — PyMuPDF native table detection
        for tbl in page.find_tables():
            clean = [[c if c else "" for c in row] for row in tbl.extract()]
            tables.append(clean)

        # Page render (captures charts that are pure graphics)
        page_png = IMG_DIR / f"pymupdf_p{page_num:03d}_{pdf_path.stem}.png"
        page.get_pixmap(matrix=fitz.Matrix(2, 2)).save(str(page_png))
        image_paths.append(str(page_png))

        # Embedded raster images inside the PDF
        for idx, img_info in enumerate(page.get_images(full=True)):
            xref     = img_info[0]
            base_img = doc.extract_image(xref)
            ext      = base_img["ext"]
            emb_path = IMG_DIR / f"pymupdf_p{page_num:03d}_img{idx}.{ext}_{pdf_path.stem}"
            emb_path.write_bytes(base_img["image"])

    doc.close()

    return {
        "text"       : "\\n".join(plain_parts),
        "markdown"   : markdown,
        "tables"     : tables,
        "image_paths": image_paths,
        "pages"      : len(plain_parts),
        "elapsed"    : time.perf_counter() - t0,
        "memory_mb"  : proc.memory_info().rss / (1024**2) - mem0,
        "cost"       : 0.0,
    }

# ── Run on every PDF in corpus/ ───────────────────────────────
pymupdf_results = {}
for pdf_path in PDF_FILES:
    print(f"  → {pdf_path.name} ...", end=" ", flush=True)
    pymupdf_results[pdf_path.stem] = run_pymupdf4llm(pdf_path)
    r = pymupdf_results[pdf_path.stem]
    print(f"{r['pages']}p  {r['elapsed']:.1f}s  "
          f"{len(r['tables'])} tables  {len(r['image_paths'])} imgs")

print("\\n✅ PyMuPDF4LLM extraction complete")

2026-05-28 10:26:17.073573833 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


  → 1_Rapport-financier-semestriel-AFD-2025.pdf ... === Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=25/26.

53p  54.9s  56 tables  53 imgs
  → 2_perfect-pitch-deck.pdf ... === Document parser messages ===
                                                                                      Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=11/12.
OCR on page.number=12/13.
OCR on page.number=13/14.
OCR on page.number=14/15.

16p  46.2s  0 tables  16 imgs
  → 3_NVCA-Model-Voting-Agreement.pdf ... === Document parser messages ===
                                                                                                                                    

Image too small to scale!! (2x36 vs min width of 3)
Line cannot be recognized!!
Image too small to scale!! (2x36 vs min width of 3)
Line cannot be recognized!!
Director exception handler, message is:
Director error: <class 'KeyboardInterrupt'>: 
Traceback (most recent call last):
    /usr/local/python/3.12.1/lib/python3.12/site-packages/pymupdf/__init__.py:23459:tell(): def tell( self, ctx):
    /usr/local/python/3.12.1/lib/python3.12/site-packages/pymupdf/mupdf.py:54672:fz_write_pixmap_as_pdfocr(): return _mupdf.fz_write_pixmap_as_pdfocr(out, pixmap, options)
    /usr/local/python/3.12.1/lib/python3.12/site-packages/pymupdf/__init__.py:13736:pdfocr_save(): mupdf.fz_write_pixmap_as_pdfocr( out, pix, opts)
    /usr/local/python/3.12.1/lib/python3.12/site-packages/pymupdf/__init__.py:13758:pdfocr_tobytes(): self.pdfocr_save(bio, compress=compress, language=language, tessdata=tessdata)
    /usr/local/python/3.12.1/lib/python3.12/site-packages/pymupdf4llm/ocr/tesseract_api.py:91:exec_ocr()

FzErrorGeneric: code=1: Director error: <class 'KeyboardInterrupt'>: 
Traceback (most recent call last):
    /usr/local/python/3.12.1/lib/python3.12/site-packages/pymupdf/__init__.py:23459:tell(): def tell( self, ctx):
    /usr/local/python/3.12.1/lib/python3.12/site-packages/py